# EA1 — Carga, transformación y resumen de un dataset

Este notebook carga **scikit-learn/iris** desde Hugging Face con `HFDatasetLoader`, aplica cuatro transformaciones y genera `reports/dataset_summary.txt` con `DatasetSummarizer`. Si Hugging Face no está disponible, usa el CSV incluido para que el flujo pueda demostrarse sin conexión.

## 1. Preparar e importar las clases

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from hf_dataset_loader import HFDatasetLoader
from dataset_summarizer import DatasetSummarizer

print("Carpeta del proyecto: raíz del repositorio")

Carpeta del proyecto: raíz del repositorio


## 2. Cargar el DataFrame desde Hugging Face

In [2]:
loader = HFDatasetLoader(
    dataset_id="scikit-learn/iris",
    split="train",
    offline_fallback_path=PROJECT_ROOT / "data" / "iris.csv",
)
df_original = loader.load()
print(f"Fuente usada: {loader.last_source}")
print(f"Dimensiones originales: {df_original.shape[0]} filas x {df_original.shape[1]} columnas")
print(df_original.head().to_string(index=False))

Fuente usada: Hugging Face: scikit-learn/iris [train]
Dimensiones originales: 150 filas x 6 columnas
 Id  SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm     Species
  1            5.1           3.5            1.4           0.2 Iris-setosa
  2            4.9           3.0            1.4           0.2 Iris-setosa
  3            4.7           3.2            1.3           0.2 Iris-setosa
  4            4.6           3.1            1.5           0.2 Iris-setosa
  5            5.0           3.6            1.4           0.2 Iris-setosa


## 3. Aplicar cuatro transformaciones

1. Traducir y estandarizar nombres de columnas.
2. Crear `area_petalo_cm2` multiplicando largo por ancho del pétalo.
3. Filtrar flores con largo de pétalo mayor a 1.5 cm.
4. Seleccionar y ordenar las columnas que se conservarán.

In [3]:
# Transformación 1: estandarizar y traducir los nombres de columnas.
standardized = df_original.rename(columns={
    "sepal length (cm)": "sepal_length_cm",
    "SepalLengthCm": "sepal_length_cm",
    "sepal width (cm)": "sepal_width_cm",
    "SepalWidthCm": "sepal_width_cm",
    "petal length (cm)": "petal_length_cm",
    "PetalLengthCm": "petal_length_cm",
    "petal width (cm)": "petal_width_cm",
    "PetalWidthCm": "petal_width_cm",
    "variety": "species",
    "Species": "species",
    "class": "species",
    "target": "species",
})
df_renamed = standardized.rename(columns={
    "sepal_length_cm": "largo_sepalo_cm",
    "sepal_width_cm": "ancho_sepalo_cm",
    "petal_length_cm": "largo_petalo_cm",
    "petal_width_cm": "ancho_petalo_cm",
    "species": "especie",
})
required = {"largo_sepalo_cm", "largo_petalo_cm", "ancho_petalo_cm", "especie"}
missing = required.difference(df_renamed.columns)
if missing:
    raise ValueError(f"No se encontraron las columnas esperadas: {sorted(missing)}")

# Transformación 2: agregar una medida derivada.
df_with_area = df_renamed.assign(
    area_petalo_cm2=df_renamed["largo_petalo_cm"] * df_renamed["ancho_petalo_cm"]
)

# Transformación 3: filtrar por una condición sencilla.
df_filtered = df_with_area.loc[df_with_area["largo_petalo_cm"] > 1.5].copy()

# Transformación 4: seleccionar las columnas de salida.
df_transformed = df_filtered[[
    "largo_sepalo_cm", "largo_petalo_cm", "ancho_petalo_cm",
    "especie", "area_petalo_cm2",
]].copy()

print(f"Filas: {len(df_original)} -> {len(df_transformed)} después del filtro")
print(f"Columnas finales: {list(df_transformed.columns)}")
print(df_transformed.head(10).to_string(index=False))

Filas: 150 -> 113 después del filtro
Columnas finales: ['largo_sepalo_cm', 'largo_petalo_cm', 'ancho_petalo_cm', 'especie', 'area_petalo_cm2']
 largo_sepalo_cm  largo_petalo_cm  ancho_petalo_cm     especie  area_petalo_cm2
             5.4              1.7              0.4 Iris-setosa             0.68
             4.8              1.6              0.2 Iris-setosa             0.32
             5.7              1.7              0.3 Iris-setosa             0.51
             5.4              1.7              0.2 Iris-setosa             0.34
             5.1              1.7              0.5 Iris-setosa             0.85
             4.8              1.9              0.2 Iris-setosa             0.38
             5.0              1.6              0.2 Iris-setosa             0.32
             5.0              1.6              0.4 Iris-setosa             0.64
             4.7              1.6              0.2 Iris-setosa             0.32
             4.8              1.6              0.2 Iris-s

## 4. Generar y mostrar el resumen en TXT

In [4]:
summary_path = PROJECT_ROOT / "reports" / "dataset_summary.txt"
summarizer = DatasetSummarizer(df_transformed)
created_file = summarizer.save(summary_path)
print(f"Resumen guardado en: {created_file.relative_to(PROJECT_ROOT)}")
print()
print(created_file.read_text(encoding="utf-8"))

Resumen guardado en: reports\dataset_summary.txt

RESUMEN DEL DATASET
Filas: 113
Columnas: 5

COLUMNAS, TIPOS Y VALORES NULOS
------------------------------------------------------------------------
- largo_sepalo_cm: tipo=float64; nulos=0
- largo_petalo_cm: tipo=float64; nulos=0
- ancho_petalo_cm: tipo=float64; nulos=0
- especie: tipo=str; nulos=0
- area_petalo_cm2: tipo=float64; nulos=0

ESTADÍSTICAS DESCRIPTIVAS
------------------------------------------------------------------------
        largo_sepalo_cm  largo_petalo_cm  ancho_petalo_cm          especie  area_petalo_cm2
count        113.000000       113.000000       113.000000              113       113.000000
unique              NaN              NaN              NaN                3              NaN
top                 NaN              NaN              NaN  Iris-versicolor              NaN
freq                NaN              NaN              NaN               50              NaN
mean           6.124779         4.534513        

## Explicación breve

Se tradujeron y estandarizaron los nombres de las columnas, se calculó el área aproximada del pétalo, se conservaron flores con largo de pétalo superior a 1.5 cm y se seleccionaron cinco columnas para el resultado. El TXT resume **el DataFrame transformado**, no el original.